# Engineering Drawing Tutor
**Gemma 4 Good Hackathon** | Theme: Education in Underserved Communities

Fine-tuned Gemma 4 E4B that reads engineering drawings, generates manufacturing
checklists, explains tolerances, and flags AS1100 compliance issues. Designed
for vocational training programs and apprentice machinists in low-resource settings.

Runs fully offline on consumer hardware (T4 GPU). No cloud dependency.

In [ ]:
# Install dependencies
!pip install unsloth[colab-new] trl datasets -q
!git clone https://github.com/govindrathore27/gemma4-engineering-diagrams.git /kaggle/working/repo 2>/dev/null || git -C /kaggle/working/repo pull
import sys
sys.path.insert(0, "/kaggle/working/repo")
print("Setup complete")

In [ ]:
# Train the Engineering Drawing Tutor model
import os
os.environ["WANDB_DISABLED"] = "true"

from shared.model.train import TrainConfig, train

cfg = TrainConfig(
    project="drawing",
    data_path="/kaggle/input/gemma4-drawing-train-data/train.jsonl",
    output_dir="/kaggle/working/drawing_adapter",
    epochs=3,
    batch_size=2,
    grad_accum=8,
)
train(cfg)
print("Training complete!")

In [ ]:
# Evaluate on held-out eval set
import json
from shared.model.inference import load_model, run
from shared.eval.metrics import evaluate_batch

model, tokenizer = load_model("/kaggle/working/drawing_adapter")
print("Model loaded")

eval_rows = [json.loads(l) for l in open("/kaggle/input/gemma4-drawing-train-data/eval.jsonl", encoding="utf-8")]
eval_rows = eval_rows[:100]

predictions = [run(model, tokenizer, r["instruction"], r["input"]) for r in eval_rows]
expected = [r["output"] for r in eval_rows]

em = evaluate_batch(predictions, expected, metric="exact_match")
f1 = evaluate_batch(predictions, expected, metric="f1")
print(f"Exact Match: {em['mean']:.3f}")
print(f"Token F1: {f1['mean']:.3f}")

with open("/kaggle/working/eval_results.txt", "w") as f:
    f.write(f"Exact Match: {em['mean']:.3f}\n")
    f.write(f"Token F1: {f1['mean']:.3f}\n")
    f.write(f"Evaluated on {len(eval_rows)} samples\n")

In [ ]:
# Export to GGUF for offline deployment
from shared.model.quantize import export_gguf
export_gguf(
    adapter_dir="/kaggle/working/drawing_adapter",
    output_path="/kaggle/working/drawing_q4km.gguf",
    quant_type="q4_k_m"
)
print("GGUF export complete")

In [ ]:
# Demo inference - Engineering Drawing Tutor
sample_drawing_context = """
Drawing: Shaft assembly, AS1100 standard
Dimensions detected: 8 dimension tags, 5 text labels
Key features: shaft diameter 25.00mm, keyway 6x6mm, overall length 120mm
"""

questions = [
    "List the critical dimensions for a shaft drawing.",
    "Create a manufacturing checklist for this engineering drawing.",
    "How do you verify a drawing conforms to AS1100 standard?",
    "What does a surface finish callout of Ra 1.6 mean?",
    "How do you read a tolerance of 25.00 ± 0.05?",
]

print("=== Engineering Drawing Tutor Demo ===\n")
for q in questions:
    answer = run(model, tokenizer, q, sample_drawing_context)
    print(f"Q: {q}")
    print(f"A: {answer}\n")

## Real-World Impact

Apprentice machinists and vocational students in developing countries often receive
engineering drawings they cannot fully interpret without expert guidance. This model
provides instant, plain-language explanations of drawing conventions, tolerances,
and manufacturing checklists — functioning as a pocket-sized tutor.

**Key capabilities:**
- Identifies critical dimensions and tolerance requirements
- Generates step-by-step manufacturing checklists
- Explains AS1100 compliance requirements
- Answers questions about GD&T, surface finish, and projection standards

**Deployment:** Runs fully offline on a T4 GPU (15 GB VRAM). GGUF export enables
CPU-only inference via llama.cpp — suitable for workshops with no internet access.

**Datasets used:**
- AS1100 Engineering Drawings (HuggingFace jcrzd): 24 annotated drawings with QA pairs
- Drawing Annotation Recognition (Roboflow): YOLO-annotated dimension tags
- Training: 1,200 QA pairs (24 gold + 1,476 synthetic templates)